In [ ]:
!pip install -q kaggle

In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR']= '/content'

In [ ]:
!kaggle datasets download -d disisbig/telugu-wikipedia-articles

100% 199M/199M [00:01<00:00, 180MB/s]
100% 199M/199M [00:01<00:00, 179MB/s]


In [ ]:
!unzip /content/data.zip -d dataset

Archive:  /content/data.zip
   creating: dataset/data/train/
  inflating: dataset/data/train/0.txt  
  inflating: dataset/data/train/1.txt  
  inflating: dataset/data/train/10.txt  
 extracting: dataset/data/train/101.txt  
  inflating: dataset/data/train/102.txt  
  inflating: dataset/data/train/104.txt  
  inflating: dataset/data/train/105.txt  
  inflating: dataset/data/train/106.txt  
  inflating: dataset/data/train/107.txt  
  inflating: dataset/data/train/108.txt  
  inflating: dataset/data/train/109.txt  
  inflating: dataset/data/train/11.txt  
  inflating: dataset/data/train/110.txt  
  inflating: dataset/data/train/111.txt  
  inflating: dataset/data/train/112.txt  
  inflating: dataset/data/train/113.txt  
  inflating: dataset/data/train/114.txt  
  inflating: dataset/data/train/116.txt  
  inflating: dataset/data/train/12.txt  
  inflating: dataset/data/train/13.txt  
  inflating: dataset/data/train/14.txt  
  inflating: dataset/data/train/15.txt  
  inflating: dataset/data

In [ ]:
!pip install transformers sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 32.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 49.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 60.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 MB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━

In [ ]:
import os
import nltk
import torch
import numpy as np
from transformers import T5Tokenizer, T5Model
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK data if not already downloaded
nltk.download('punkt')

# Load Telugu T5 tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-base")

# Load Telugu T5 model
model = T5Model.from_pretrained("t5-base")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [ ]:
import os
import nltk
import torch
import numpy as np
from transformers import T5Tokenizer, T5Model
from sklearn.metrics.pairwise import cosine_similarity

# Function to generate sentence embeddings using T5 model
def generate_sentence_embeddings(sentences):
    embeddings = []
    for sentence in sentences:
        input_ids = tokenizer.encode(sentence, add_special_tokens=True, return_tensors="pt", max_length=512, truncation=True)
        with torch.no_grad():
            outputs = model(input_ids=input_ids, decoder_input_ids=input_ids)
            sentence_embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
            embeddings.append(sentence_embedding)
    return np.array(embeddings)

# Function to calculate similarity matrix
def calculate_similarity_matrix(embeddings):
    similarity_matrix = np.dot(embeddings, embeddings.T)
    return similarity_matrix

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Summary for text 1:
3. 1. 3.

Summary for text 2:
అధిక తేమ చర్మం నుండి తేమ యొక్క ఆవిరి రేటు తగ్గించడం ద్వారా శరీరం శీతలీకరణలో చెమట పట్టుట యొక్క ప్రభావంను తగ్గిస్తుంది. పరమ ఆర్ద్రత 2. ప్రస్తుత ఉష్ణోగ్రత వద్ద గాలిలో ఉండే తేమను సంతృప్తీకరణం చేయడానికి కావలసిన తేమ శాతాన్ని సాపేక్ష ఆర్ద్రత అంటారు.

Summary for text 3:
కిస్మత్‌పూర్,తెలంగాణ రాష్ట్రం, రంగారెడ్డి జిల్లా, గండిపేట్ మండలానికి చెందిన గ్రామం.
సముద్రమట్టానికి 556 మీ.ఎత్తు Time zone: IST 
2011 భారత జనగణన గణాంకాల ప్రకారం గ్రామ పరిధిలోని జనాభా- మొత్తం 	7,288 - పురుషుల సంఖ్య 3,693 - స్త్రీల సంఖ్య3,595 - గృహాల సంఖ్య 1,547
న్యూ లిటిల్ స్కాలర్సు హైస్కూల్, మారికా హైస్కూల్, రాక్ హైస్కూల్, కె.ఎజి.బ్.వి.స్కూల్, కిస్మత్‌పూర్
సిటీబస్సు సౌకర్యం కలదు. మేజర్ రైల్వే స్టేషన్ హైదరాబాదు 12 కి.మీ


Summary for text 4:
2011 భారత జనగణన గణాంకాల ప్రకారం ఈ గ్రామం 4662 ఇళ్లతో, 22270 జనాభాతో 4736 హెక్టార్లలో విస్తరించి ఉంది. 2001 వ.సంవత్సరం జనాభా లెక్కల ప్రకారం గ్రామ జనాభా 19,805. ఒక ప్రభుత్వ జూనియర్ కళాశాల ఉంది.సమీప ప్రభుత్వ ఆర్ట్స్ / సైన్స్ డిగ్రీ కళాశాల, ఇంజన

In [ ]:

# Function to generate extractive summary from Telugu text using T5 model
def generate_extractive_summary_t5(text, num_sentences=3):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text
    embeddings = generate_sentence_embeddings(sentences)
    similarity_matrix = calculate_similarity_matrix(embeddings)
    # Initialize score for each sentence
    scores = np.zeros(len(sentences))
    # Calculate score for each sentence based on similarity with other sentences
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                scores[i] += similarity_matrix[i][j]
    # Get indices of top scoring sentences
    top_sentences_indices = np.argsort(scores)[::-1][:num_sentences]
    # Sort indices to maintain original order
    top_sentences_indices = sorted(top_sentences_indices)
    # Generate extractive summary
    summary = ' '.join([sentences[i] for i in top_sentences_indices])
    return summary


In [ ]:
# Path to the directory containing Telugu text files
train_data_path = "/content/dataset/data/train"

# Function to read Telugu text files from directory
def read_telugu_text_files(directory):
    texts = []
    for filename in os.listdir(directory):
        with open(os.path.join(directory, filename), 'r', encoding='utf-8') as file:
            texts.append(file.read())
    return texts


In [ ]:

# Read Telugu text data from train directory
telugu_texts = read_telugu_text_files(train_data_path)

# Generate summaries for each Telugu text in train data
for i, text in enumerate(telugu_texts):
    summary = generate_extractive_summary_t5(text)
    print(f"Summary for text {i+1}:\n{summary}\n")

In [ ]:
!pip install rouge

In [ ]:
import os
import nltk
import torch
import numpy as np
from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge
from transformers import T5Tokenizer, T5Model

# Download NLTK data if not already downloaded
nltk.download('punkt')

# Initialize Telugu T5 tokenizer and model
tokenizer = T5Tokenizer.from_pretrained("t5-base")
model = T5Model.from_pretrained("t5-base")

# Function to generate sentence embeddings using T5 model
def generate_sentence_embeddings(sentences):
    embeddings = []
    for sentence in sentences:
        input_ids = tokenizer.encode(sentence, add_special_tokens=True, return_tensors="pt", max_length=512, truncation=True)
        with torch.no_grad():
            outputs = model(input_ids=input_ids, decoder_input_ids=input_ids)
            sentence_embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
            embeddings.append(sentence_embedding)
    return np.array(embeddings)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
# Function to calculate similarity matrix
def calculate_similarity_matrix(embeddings):
    similarity_matrix = np.dot(embeddings, embeddings.T)
    return similarity_matrix

# Function to generate extractive summary from Telugu text using T5 model
def generate_extractive_summary_t5(text, num_sentences=3):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text
    embeddings = generate_sentence_embeddings(sentences)
    similarity_matrix = calculate_similarity_matrix(embeddings)
    # Initialize score for each sentence
    scores = np.zeros(len(sentences))
    # Calculate score for each sentence based on similarity with other sentences
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                scores[i] += similarity_matrix[i][j]
    # Get indices of top scoring sentences
    top_sentences_indices = np.argsort(scores)[::-1][:num_sentences]
    # Sort indices to maintain original order
    top_sentences_indices = sorted(top_sentences_indices)
    # Generate extractive summary
    summary = ' '.join([sentences[i] for i in top_sentences_indices])
    return summary

In [ ]:
# Function to calculate BLEU score
def calculate_bleu(reference, summary):
    reference_tokens = nltk.word_tokenize(reference.lower())
    summary_tokens = nltk.word_tokenize(summary.lower())
    return sentence_bleu([reference_tokens], summary_tokens)

# Function to calculate ROUGE score
def calculate_rouge(reference, summary):
    rouge = Rouge()
    scores = rouge.get_scores(summary, reference)
    rouge_1_recall = scores[0]['rouge-1']['r']
    rouge_1_precision = scores[0]['rouge-1']['p']
    rouge_1_f1 = scores[0]['rouge-1']['f']
    return rouge_1_recall, rouge_1_precision, rouge_1_f1

In [ ]:
# Function to generate summary from file
def generate_summary_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    summary = generate_extractive_summary_t5(text)
    return summary

# Example file path for inference
file_path = "/content/dataset/data/valid/123.txt"

# Generate summary using inference function
generated_summary = generate_summary_from_file(file_path)

print("Generated Summary:")
print(generated_summary)

Generated Summary:
అగ్ని ప్రమాదాలు జరిగినప్పుడు మాత్రమే అత్యధికులకు గుర్తుకు వచ్చే అగ్నిమాపక శాఖను ప్రజలను మరింత చేరువచేయడానికి ప్రభుత్వం దాని పేరును మార్చింది. 'లైఫ్ సేవింగ్ జాకెట్ల ప్రజలకివ్వాలి. 101 నెంబరుకు ఫోన్ చేస్తే శాఖాపరంగా బాధితులకు అవసరమైన సేవలు అందిస్తారు


In [ ]:
# Read reference text from file
with open(file_path, 'r', encoding='utf-8') as file:
    reference_text = file.read()

# Calculate BLEU score
bleu_score = calculate_bleu(reference_text, generated_summary)
print("BLEU Score:", bleu_score)

# Calculate ROUGE score
rouge_recall, rouge_precision, rouge_f1 = calculate_rouge(reference_text, generated_summary)
print("ROUGE-1 Recall:", rouge_recall)
print("ROUGE-1 Precision:", rouge_precision)
print("ROUGE-1 F1 Score:", rouge_f1)


BLEU Score: 0.003636065025044153
ROUGE-1 Recall: 0.1895424836601307
ROUGE-1 Precision: 1.0
ROUGE-1 F1 Score: 0.31868131600229443


In [ ]:
import nltk
from transformers import T5Tokenizer, T5Model
import torch

# Download NLTK data if not already downloaded
nltk.download('punkt')

# Initialize Telugu T5 tokenizer and model
tokenizer = T5Tokenizer.from_pretrained("t5-base")
model = T5Model.from_pretrained("t5-base")

# Function to generate extractive summary from Telugu text using T5 model
def generate_extractive_summary_t5(text, num_sentences=3):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text
    input_text = ' '.join(sentences[:5])  # Select first 5 lines
    input_ids = tokenizer.encode(input_text, add_special_tokens=True, return_tensors="pt", max_length=512, truncation=True)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, decoder_input_ids=input_ids)
        summary_tokens = outputs[0][0].argmax(dim=-1).cpu().numpy()
    summary = tokenizer.decode(summary_tokens, skip_special_tokens=True)
    return summary

# Example Telugu text for inference
telugu_text = "సమీక్ష దివియా బలనింది, దేవి దొంగలు, హైదరాబాద్ టెన్నీస్‌కి బీడీఆర్‌బారు ముందుకు ప్రపంచకు నిర్మాణం చేశారు.ఎంతో హిట్లర్ లు నవీన తెలుగు చిత్ర దర్శకుడు వందనా విత్‌డి వందనా విత్‌కు వీపెట్టింది.చిరంజీవి జరగని ఆరోగ్య సమస్యలను తగ్గించే అంశాలు ముఖ్యంగా ముగించారు.భారత్ బ్యాడ్‌మింటన్ వెళ్ళిపోయింది, ఆకాశ్‌ను పట్టిస్తున్న అస్ట్రోనాట్స్ మనకు చెందండి.దిల్‌హీలో మస్జిద్‌ను దాఖలు చేశారు, భారత్ రిపబ్లిక్‌ను కోటి సమాచారం నిర్వహించడం కోరిక."


# Generate extractive summary using T5 model
extractive_summary_t5 = generate_extractive_summary_t5(telugu_text)

print("Extractive Summary (T5 Model):")
print(extractive_summary_t5)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Extractive Summary (T5 Model):
సమీక్ష దివియా బలనింది, దేవి దొంగలు, హైదరాబాద్ టెన్నీస్‌కి బీడీఆర్‌బారు ముందుకు ప్రపంచకు నిర్మాణం చేశారు.ఎంతో హిట్లర్ లు నవీన తెలుగు చిత్ర దర్శకుడు వందనా విత్‌డి వందనా విత్‌కు వీపెట్టింది.చిరంజీవి జరగని ఆరోగ్య సమస్యలను తగ్గించే అంశాలు ముఖ్యంగా ముగించారు.భారత్ బ్యాడ్‌మింటన్ వెళ్ళిపోయింది, ఆకాశ్‌ను పట్టిస్తున్న అస్ట్రోనాట్స్ మనకు చెందండి.దిల్‌హీలో మస్జిద్‌ను దాఖలు చేశారు, భారత్ రిపబ్లిక్‌ను కోటి సమాచారం నిర్వహించడం కోరిక.
